# EDA workspace

In [ ]:
import os
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("project-eda")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.caseSensitive", "true")
         .getOrCreate())
dataset = os.getenv("DATASET_NAME", "video_games_metadata")
version = os.getenv("DATASET_VERSION", "v1")
input_path = f"hdfs://namenode:8020/project/raw/{dataset}/{version}"
print(input_path)

In [ ]:
from pyspark.sql import functions as F

# Đọc các file Flume đã ghi trong HDFS
path = "hdfs://namenode:8020/project/raw/video_games_metadata/v1"
df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(path)
)

# Xem cấu trúc dữ liệu và vài bản ghi
df.printSchema()
df.show(10, truncate=False)

# Số bản ghi và số cột
print("Rows:", df.count())
print("Columns:", len(df.columns))

# Kiểm tra số lượng giá trị null theo cột
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(truncate=False)

# Thống kê các cột số
df.describe().show(truncate=False)

# Đếm giá trị phổ biến của một cột, ví dụ Genre
df.groupBy("Genre").count().orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
spark.conf.set("spark.sql.caseSensitive", "true")

path = "hdfs://namenode:8020/project/raw/video_games_metadata/v1"

df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(path)
)

df.printSchema()
df.show(5, truncate=False)